# Minggu 3 — Missing, duplikat, outlier

**Sub-CPMK:** **P3**

Alur konsisten dengan modul PDF: dataset **titanic** (Seaborn). Dokumentasikan keputusan cleaning di Markdown.

**Audit missing.** `load_dataset(...).copy()` membuat salinan agar `df_raw` tidak berbagi memori dengan `df`. `isna().sum()` menghitung NA per kolom; pengurutan menonjolkan kolom bermasalah terbesar.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

df = sns.load_dataset("titanic").copy()  # salinan independen untuk transformasi
df_raw = df.copy()  # cadangan untuk dokumentasi "sebelum cleaning"

missing = df.isna().sum().sort_values(ascending=False)  # jumlah NA per kolom, terbesar dulu
missing[missing > 0]  # tampilkan hanya kolom yang punya missing

**Imputasi dan penyesuaian kolom.** Menghapus `deck` yang sangat sparse; `fillna(median)` untuk `age`; modus untuk `embarked`. Loop numerik mengisi sisa NA dengan median per kolom (contoh agresif—dokumentasikan trade-off di Markdown).

In [ ]:
df_clean = df.copy()
df_clean = df_clean.drop(columns=["deck"], errors="ignore")  # kolom terlalu banyak NA untuk contoh ini

df_clean["age"] = df_clean["age"].fillna(df_clean["age"].median())  # imputasi numerik robust
emb_mode = df_clean["embarked"].mode()  # kategori paling sering
if len(emb_mode) > 0:
    df_clean["embarked"] = df_clean["embarked"].fillna(emb_mode[0])

num_cols = df_clean.select_dtypes(include="number").columns  # semua kolom numerik
for c in num_cols:
    med = df_clean[c].median()
    df_clean[c] = df_clean[c].fillna(med)

**Duplikat.** `drop_duplicates()` menghapus baris identik; membandingkan `shape` sebelum/sesudah memberi bukti di laporan praktikum.

In [ ]:
print("sebelum", df_clean.shape)
df_clean = df_clean.drop_duplicates()  # hapus baris duplikat penuh
print("sesudah", df_clean.shape)

**Visual outlier.** Boxplot `fare` per `class` menunjukkan median, kuartil, dan titik luar—input untuk keputusan winsorize, transform log, atau membiarkan model yang robust.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.boxplot(data=df_clean, x="class", y="fare")  # distribusi fare per kelas kabin
plt.title("Sebaran fare per class (deteksi outlier)")
plt.tight_layout()
plt.show()

## Latihan
- Tulis paragraf *trade-off* shape sebelum/sesudah cleaning.
- (Opsional) batasi outlier `fare` dengan IQR dan bandingkan ringkasan deskriptif.